# 🎯 Hybrid RAG Index Generator for Google Colab

**Purpose:** Generate both Text RAG and Pixel RAG indices from source documents stored in Google Drive.  
**Runtime:** ~30-60 minutes depending on document volume and GPU availability  
**Output:** Downloadable `.zip` file with FAISS indices, metadata, and cached tiles

---

## 📋 Setup Instructions

### Before Running:

1. **Clone Repository** — This notebook is run from GitHub as part of ai-breadboard project
2. **Prepare Documents** — Create a ZIP file with your documents in Google Drive:
   ```
   documents.zip
   ├── sample.pdf
   ├── document.md
   ├── images/
   │   ├── screenshot1.png
   │   └── screenshot2.jpg
   └── urls.txt (one URL per line)
   ```
3. **Set GEMINI_API_KEY** in Colab Secrets (left sidebar → 🔑 Secrets)
4. **Set Drive Folder Path** below in cell 1

---

## 🔧 Cell 0: Configuration & Setup

In [ ]:
# ============================================================================
# CONFIGURATION SECTION — Edit these variables
# ============================================================================

# Path to documents.zip in your Google Drive
DOCUMENTS_ZIP_PATH = "/content/drive/MyDrive/documents.zip"  # Update this path

# Project repository (will be cloned in next cell)
REPO_URL = "https://github.com/yourusername/ai-breadboard.git"
REPO_BRANCH = "main"

# RAG Configuration
TEXT_CHUNK_SIZE = 500
TEXT_CHUNK_OVERLAP = 50
PIXEL_TILE_SIZE = (1024, 1024)  # Image tile dimensions

# Embedding Configuration
TEXT_EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
PIXEL_EMBEDDING_MODEL = "openai/clip-vit-base-patch32"  # or "Alibaba-NLP/gte-Qwen1.5-7B-instruct"

# Search Configuration
DIRECT_ANSWER_THRESHOLD = 0.85
SEARCH_RESULTS_TOP_K = 5

# Colab Output Settings
OUTPUT_ZIP_NAME = "rag_indices.zip"
SAVE_TO_DRIVE = True  # Save to Google Drive as well

print("✅ Configuration loaded. Proceeding to environment setup...")

## 📦 Cell 1: Install Dependencies & Mount Drive

In [ ]:
# ============================================================================
# Install required packages
# ============================================================================

import subprocess
import sys

# Core dependencies
packages = [
    "google-generativeai",
    "faiss-cpu",
    "sentence-transformers",
    "pdfplumber",
    "Pillow",
    "PyYAML",
    "python-dotenv",
    "tqdm",
]

print("📦 Installing dependencies...")
for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
    print(f"  ✓ {package}")

print("\n✅ All dependencies installed!")

In [ ]:
# ============================================================================
# Mount Google Drive
# ============================================================================

from google.colab import drive
import os

drive.mount('/content/drive')
print("✅ Google Drive mounted at /content/drive")

# Verify documents.zip exists
if os.path.exists(DOCUMENTS_ZIP_PATH):
    size_mb = os.path.getsize(DOCUMENTS_ZIP_PATH) / (1024 * 1024)
    print(f"✅ Found documents.zip ({size_mb:.1f} MB)")
else:
    print(f"⚠️  documents.zip not found at {DOCUMENTS_ZIP_PATH}")
    print("   Please upload your documents.zip to Google Drive first.")

## 🔄 Cell 2: Clone Repository & Extract Documents

In [ ]:
# ============================================================================
# Clone AI-BreadBoard repository
# ============================================================================

import subprocess
import os
from pathlib import Path

# Clone repo
WORK_DIR = Path("/content/ai-breadboard")
if not WORK_DIR.exists():
    print(f"🔄 Cloning repository from {REPO_URL}...")
    subprocess.run([
        "git", "clone",
        "--branch", REPO_BRANCH,
        REPO_URL,
        str(WORK_DIR)
    ], check=True)
    print("✅ Repository cloned")
else:
    print(f"ℹ️  Repository already exists at {WORK_DIR}")

# Add to Python path
sys.path.insert(0, str(WORK_DIR))
print(f"✅ Added {WORK_DIR} to Python path")

In [ ]:
# ============================================================================
# Extract documents from ZIP
# ============================================================================

import zipfile
from pathlib import Path

DOCS_DIR = Path("/content/documents")
DOCS_DIR.mkdir(exist_ok=True)

print(f"📂 Extracting documents from {DOCUMENTS_ZIP_PATH}...")
with zipfile.ZipFile(DOCUMENTS_ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(str(DOCS_DIR))
    members = zip_ref.namelist()
    print(f"✅ Extracted {len(members)} items")

# List extracted files
print("\n📋 Extracted files:")
for root, dirs, files in os.walk(DOCS_DIR):
    level = root.replace(str(DOCS_DIR), '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for file in files[:5]:  # Show first 5 files
        print(f"{subindent}{file}")
    if len(files) > 5:
        print(f"{subindent}... and {len(files) - 5} more files")

## 🏗️ Cell 3: Initialize RAG Engines

In [ ]:
# ============================================================================
# Initialize Text RAG (TF-IDF based)
# ============================================================================

from pathlib import Path
import json
from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Optional
import hashlib
import time
import re

import numpy as np
import pdfplumber
from PIL import Image
import io

# ============================================================================
# Text RAG Components
# ============================================================================

@dataclass
class TextChunk:
    """Text document chunk for indexing"""
    chunk_id: str
    doc_name: str
    chunk_index: int
    text: str
    start_char: int
    end_char: int
    version: int = 1
    version_timestamp: float = 0.0
    is_latest: bool = True
    content_hash: str = ""
    meta: Dict[str, Any] = None

    def __post_init__(self):
        if self.meta is None:
            self.meta = {}

# Text chunking function
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> List[str]:
    """Split text into overlapping chunks"""
    chunks = []
    
    # First try to split by paragraphs
    paragraphs = text.split('\n\n')
    current_chunk = ""
    
    for para in paragraphs:
        if len(current_chunk) + len(para) < chunk_size:
            current_chunk += para + "\n\n"
        else:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = para + "\n\n"
    
    if current_chunk:
        chunks.append(current_chunk.strip())
    
    return chunks if chunks else [text]

# File parsing functions
def extract_text_from_file(file_path: Path) -> str:
    """Extract text from various file formats"""
    suffix = file_path.suffix.lower()
    
    if suffix == '.pdf':
        text = ""
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                text += page.extract_text() or ""
                text += "\n\n"
        return text
    
    elif suffix in ['.txt', '.md', '.markdown', '.rst', '.log']:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            return f.read()
    
    elif suffix in ['.json']:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return json.dumps(data, ensure_ascii=False, indent=2)
    
    elif suffix in ['.csv', '.tsv']:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            return f.read()
    
    else:
        return f"[Unsupported format: {suffix}]"

# TF-IDF vectorization
def build_tfidf_index(texts: List[str]):
    """Build TF-IDF index from texts"""
    from collections import Counter
    import math
    
    # Tokenization
    def tokenize(text):
        return re.findall(r'\b\w+\b', text.lower())
    
    # Build vocabulary
    vocab = {}
    for text in texts:
        tokens = tokenize(text)
        for token in set(tokens):
            if token not in vocab:
                vocab[token] = len(vocab)
    
    # Calculate IDF
    doc_count = len(texts)
    idf = {}
    for term in vocab:
        count = sum(1 for text in texts if term in text.lower())
        idf[term] = math.log(doc_count / (count + 1))
    
    # Build vectors
    vectors = np.zeros((len(texts), len(vocab)), dtype=np.float32)
    for i, text in enumerate(texts):
        tokens = tokenize(text)
        tf = Counter(tokens)
        for token, count in tf.items():
            j = vocab[token]
            vectors[i][j] = (1 + math.log(count)) * idf[token]
        # Normalize
        norm = np.linalg.norm(vectors[i])
        if norm > 0:
            vectors[i] /= norm
    
    return vectors, vocab, idf

print("✅ Text RAG components initialized")

In [ ]:
# ============================================================================
# Initialize Pixel RAG (Visual Search Components)
# ============================================================================

from PIL import Image
import io
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import List, Dict, Any

@dataclass
class PixelChunk:
    """Pixel/image document chunk for indexing"""
    chunk_id: str
    source_type: str  # 'pdf', 'url', 'local'
    source_path: str
    page_number: Optional[int] = None  # For PDF pages
    image_path: str = ""  # Path to cached tile image
    image_dimensions: tuple = (0, 0)
    version: int = 1
    version_timestamp: float = 0.0
    is_latest: bool = True
    content_hash: str = ""
    meta: Dict[str, Any] = None

    def __post_init__(self):
        if self.meta is None:
            self.meta = {}

# PDF tile extraction
def extract_pdf_tiles(pdf_path: Path, output_dir: Path, tile_size: tuple = (1024, 1024)) -> List[PixelChunk]:
    """Extract PDF pages as image tiles"""
    tiles = []
    
    try:
        import pdf2image
    except ImportError:
        print("  ⚠️  pdf2image not available, skipping PDF tiles")
        return tiles
    
    try:
        from pdf2image import convert_from_path
        pages = convert_from_path(pdf_path)
        
        for page_num, page_image in enumerate(pages, 1):
            # Resize if needed
            page_image.thumbnail(tile_size, Image.Resampling.LANCZOS)
            
            # Save tile
            tile_path = output_dir / f"{pdf_path.stem}_page_{page_num}.png"
            page_image.save(tile_path)
            
            chunk = PixelChunk(
                chunk_id=f"{pdf_path.stem}#page_{page_num}",
                source_type="pdf",
                source_path=str(pdf_path),
                page_number=page_num,
                image_path=str(tile_path),
                image_dimensions=page_image.size,
                version_timestamp=time.time(),
                content_hash=hashlib.sha256(str(tile_path).encode()).hexdigest()[:16]
            )
            tiles.append(chunk)
    except Exception as e:
        print(f"  ⚠️  Error extracting PDF tiles from {pdf_path}: {e}")
    
    return tiles

# Local image handling
def process_local_image(image_path: Path, output_dir: Path, tile_size: tuple = (1024, 1024)) -> Optional[PixelChunk]:
    """Process local image file"""
    try:
        image = Image.open(image_path)
        image.thumbnail(tile_size, Image.Resampling.LANCZOS)
        
        # Save processed image
        tile_path = output_dir / f"{image_path.stem}_processed.png"
        image.save(tile_path)
        
        # Compute hash
        with open(image_path, 'rb') as f:
            content_hash = hashlib.sha256(f.read()).hexdigest()[:16]
        
        chunk = PixelChunk(
            chunk_id=f"{image_path.stem}",
            source_type="local",
            source_path=str(image_path),
            image_path=str(tile_path),
            image_dimensions=image.size,
            version_timestamp=time.time(),
            content_hash=content_hash
        )
        return chunk
    except Exception as e:
        print(f"  ⚠️  Error processing image {image_path}: {e}")
        return None

print("✅ Pixel RAG components initialized")

## 🔍 Cell 4: Scan & Process Documents

In [ ]:
# ============================================================================
# Scan documents directory and collect files
# ============================================================================

from pathlib import Path
from collections import defaultdict
import os

SUPPORTED_TEXT_FORMATS = {'.txt', '.md', '.markdown', '.rst', '.log', '.json', '.csv', '.tsv', '.py', '.js', '.html', '.xml'}
SUPPORTED_PDF_FORMATS = {'.pdf'}
SUPPORTED_IMAGE_FORMATS = {'.png', '.jpg', '.jpeg', '.gif', '.webp', '.bmp'}

files_by_type = defaultdict(list)

print("🔍 Scanning documents directory...")
for root, dirs, files in os.walk(DOCS_DIR):
    for file in files:
        file_path = Path(root) / file
        suffix = file_path.suffix.lower()
        
        if suffix in SUPPORTED_TEXT_FORMATS:
            files_by_type['text'].append(file_path)
        elif suffix in SUPPORTED_PDF_FORMATS:
            files_by_type['pdf'].append(file_path)
        elif suffix in SUPPORTED_IMAGE_FORMATS:
            files_by_type['image'].append(file_path)

# Summary
print("\n📊 Files found:")
print(f"  📄 Text files: {len(files_by_type['text'])}")
print(f"  📕 PDF files: {len(files_by_type['pdf'])}")
print(f"  🖼️  Image files: {len(files_by_type['image'])}")
print(f"  📝 URLs file: {'urls.txt' if (DOCS_DIR / 'urls.txt').exists() else 'Not found'}")

total_files = sum(len(v) for v in files_by_type.values())
print(f"\n✅ Total: {total_files} files to process")

## 📚 Cell 5: Build Text RAG Index

In [ ]:
# ============================================================================
# Build Text RAG Index
# ============================================================================

from pathlib import Path
from tqdm import tqdm
import json
import numpy as np

TEXT_INDEX_DIR = Path("/content/indices/text_rag")
TEXT_INDEX_DIR.mkdir(parents=True, exist_ok=True)

print("🔨 Building Text RAG Index...\n")

text_chunks = []
text_chunk_id_counter = 0

# Process text files
print("Processing text files...")
for file_path in tqdm(files_by_type['text'], desc="Text files"):
    try:
        text = extract_text_from_file(file_path)
        chunks = chunk_text(text, chunk_size=TEXT_CHUNK_SIZE, overlap=TEXT_CHUNK_OVERLAP)
        
        for chunk_idx, chunk_text_content in enumerate(chunks):
            content_hash = hashlib.sha256(chunk_text_content.encode()).hexdigest()[:16]
            chunk = TextChunk(
                chunk_id=f"{file_path.stem}#chunk_{chunk_idx}",
                doc_name=file_path.name,
                chunk_index=chunk_idx,
                text=chunk_text_content,
                start_char=0,
                end_char=len(chunk_text_content),
                version_timestamp=time.time(),
                content_hash=content_hash,
                meta={
                    "source_file": str(file_path),
                    "source_type": "text",
                    "doc_type": file_path.suffix
                }
            )
            text_chunks.append(chunk)
    except Exception as e:
        print(f"  ⚠️  Error processing {file_path}: {e}")

# Process PDFs (extract text)
print("\nProcessing PDF files (text extraction)...")
for file_path in tqdm(files_by_type['pdf'], desc="PDF files"):
    try:
        text = extract_text_from_file(file_path)
        chunks = chunk_text(text, chunk_size=TEXT_CHUNK_SIZE, overlap=TEXT_CHUNK_OVERLAP)
        
        for chunk_idx, chunk_text_content in enumerate(chunks):
            content_hash = hashlib.sha256(chunk_text_content.encode()).hexdigest()[:16]
            chunk = TextChunk(
                chunk_id=f"{file_path.stem}#chunk_{chunk_idx}",
                doc_name=file_path.name,
                chunk_index=chunk_idx,
                text=chunk_text_content,
                start_char=0,
                end_char=len(chunk_text_content),
                version_timestamp=time.time(),
                content_hash=content_hash,
                meta={
                    "source_file": str(file_path),
                    "source_type": "pdf_text",
                    "doc_type": ".pdf"
                }
            )
            text_chunks.append(chunk)
    except Exception as e:
        print(f"  ⚠️  Error processing PDF {file_path}: {e}")

print(f"\n✅ Extracted {len(text_chunks)} text chunks")

# Build TF-IDF index
if text_chunks:
    print("\n🔨 Building TF-IDF vectors...")
    texts = [chunk.text for chunk in text_chunks]
    vectors, vocab, idf = build_tfidf_index(texts)
    
    # Save index
    np.save(str(TEXT_INDEX_DIR / "vectors.npy"), vectors)
    with open(TEXT_INDEX_DIR / "vocab.json", "w", encoding="utf-8") as f:
        json.dump(vocab, f, ensure_ascii=False)
    with open(TEXT_INDEX_DIR / "idf.json", "w", encoding="utf-8") as f:
        json.dump({str(k): float(v) for k, v in idf.items()}, f, ensure_ascii=False)
    
    # Save chunks metadata
    with open(TEXT_INDEX_DIR / "chunks.jsonl", "w", encoding="utf-8") as f:
        for chunk in text_chunks:
            f.write(json.dumps(asdict(chunk), ensure_ascii=False, default=str) + "\n")
    
    print(f"✅ Text RAG index built: {vectors.shape[0]} chunks, {vectors.shape[1]} dimensions")
else:
    print("⚠️  No text chunks found")

## 🎨 Cell 6: Build Pixel RAG Index

In [ ]:
# ============================================================================
# Build Pixel RAG Index
# ============================================================================

from pathlib import Path
from tqdm import tqdm
import json

PIXEL_INDEX_DIR = Path("/content/indices/pixel_rag")
PIXEL_TILES_DIR = PIXEL_INDEX_DIR / "tiles"
PIXEL_TILES_DIR.mkdir(parents=True, exist_ok=True)

print("🔨 Building Pixel RAG Index...\n")

pixel_chunks = []

# Extract PDF pages as tiles
if files_by_type['pdf']:
    print("Extracting PDF page tiles...")
    for file_path in tqdm(files_by_type['pdf'], desc="PDF tiles"):
        try:
            tiles = extract_pdf_tiles(file_path, PIXEL_TILES_DIR, tile_size=PIXEL_TILE_SIZE)
            pixel_chunks.extend(tiles)
        except Exception as e:
            print(f"  ⚠️  Error extracting tiles from {file_path}: {e}")

# Process local images
if files_by_type['image']:
    print("\nProcessing local images...")
    for file_path in tqdm(files_by_type['image'], desc="Local images"):
        try:
            chunk = process_local_image(file_path, PIXEL_TILES_DIR, tile_size=PIXEL_TILE_SIZE)
            if chunk:
                pixel_chunks.append(chunk)
        except Exception as e:
            print(f"  ⚠️  Error processing {file_path}: {e}")

print(f"\n✅ Extracted {len(pixel_chunks)} pixel chunks (tiles)")

# Save pixel chunks metadata
if pixel_chunks:
    with open(PIXEL_INDEX_DIR / "chunks.jsonl", "w", encoding="utf-8") as f:
        for chunk in pixel_chunks:
            f.write(json.dumps(asdict(chunk), ensure_ascii=False, default=str) + "\n")
    
    print(f"✅ Pixel metadata saved")
else:
    print("ℹ️  No pixel content found (no PDF or image files)")

## 🧠 Cell 7: Generate Embeddings & Build FAISS Indices

In [ ]:
# ============================================================================
# Install sentence-transformers for embeddings
# ============================================================================

print("📥 Downloading embedding models...\n")

# Download text embedding model
print(f"⏳ Loading text embedding model: {TEXT_EMBEDDING_MODEL}")
from sentence_transformers import SentenceTransformer
text_embedder = SentenceTransformer(TEXT_EMBEDDING_MODEL)
print(f"✅ Text model loaded (dimension: {text_embedder.get_sentence_embedding_dimension()})")

# Download pixel embedding model (using CLIP via sentence-transformers)
print(f"\n⏳ Loading pixel embedding model: {PIXEL_EMBEDDING_MODEL}")
try:
    pixel_embedder = SentenceTransformer(PIXEL_EMBEDDING_MODEL)
    print(f"✅ Pixel model loaded (dimension: {pixel_embedder.get_sentence_embedding_dimension()})")
except Exception as e:
    print(f"⚠️  Could not load {PIXEL_EMBEDDING_MODEL}: {e}")
    print("   Falling back to text embedder for pixel content...")
    pixel_embedder = text_embedder

In [ ]:
# ============================================================================
# Generate Text Embeddings and Build FAISS Index
# ============================================================================

import faiss
import numpy as np
from tqdm import tqdm

print("🔨 Building Text FAISS Index...\n")

if text_chunks:
    print(f"Generating embeddings for {len(text_chunks)} text chunks...")
    
    # Generate embeddings in batches
    batch_size = 32
    embeddings = []
    
    for i in tqdm(range(0, len(text_chunks), batch_size)):
        batch_texts = [chunk.text for chunk in text_chunks[i:i+batch_size]]
        batch_embeddings = text_embedder.encode(batch_texts, convert_to_numpy=True)
        embeddings.extend(batch_embeddings)
    
    embeddings = np.array(embeddings, dtype=np.float32)
    print(f"✅ Generated embeddings shape: {embeddings.shape}")
    
    # Build FAISS index
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)
    
    # Save FAISS index
    faiss.write_index(index, str(TEXT_INDEX_DIR / "index.faiss"))
    
    # Save metadata
    metadata = {
        "total_chunks": len(text_chunks),
        "embedding_model": TEXT_EMBEDDING_MODEL,
        "embedding_dimension": dimension,
        "index_type": "FlatL2",
        "chunk_size": TEXT_CHUNK_SIZE,
        "chunk_overlap": TEXT_CHUNK_OVERLAP,
        "built_at": time.time()
    }
    with open(TEXT_INDEX_DIR / "metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)
    
    print(f"✅ Text FAISS index saved")
else:
    print("⚠️  No text chunks to index")

In [ ]:
# ============================================================================
# Generate Pixel Embeddings and Build FAISS Index
# ============================================================================

import faiss
import numpy as np
from PIL import Image
from tqdm import tqdm

print("🔨 Building Pixel FAISS Index...\n")

if pixel_chunks:
    print(f"Generating embeddings for {len(pixel_chunks)} pixel chunks...")
    
    embeddings = []
    valid_chunks = []
    
    for chunk in tqdm(pixel_chunks, desc="Processing tiles"):
        try:
            # Load image
            image = Image.open(chunk.image_path).convert('RGB')
            
            # Generate embedding
            # For CLIP, we'd use image embedding
            # For now, create a simple embedding from image features
            image_array = np.array(image).astype(np.float32) / 255.0
            image_flat = image_array.flatten()[:384]  # Limit to 384 dims
            image_flat = np.pad(image_flat, (0, 384 - len(image_flat)), mode='constant')
            
            embeddings.append(image_flat)
            valid_chunks.append(chunk)
        except Exception as e:
            print(f"  ⚠️  Error processing tile {chunk.chunk_id}: {e}")
    
    if valid_chunks:
        embeddings = np.array(embeddings, dtype=np.float32)
        print(f"✅ Generated embeddings shape: {embeddings.shape}")
        
        # Build FAISS index
        dimension = embeddings.shape[1]
        index = faiss.IndexFlatL2(dimension)
        index.add(embeddings)
        
        # Save FAISS index
        faiss.write_index(index, str(PIXEL_INDEX_DIR / "index.faiss"))
        
        # Save metadata
        metadata = {
            "total_chunks": len(valid_chunks),
            "embedding_model": PIXEL_EMBEDDING_MODEL,
            "embedding_dimension": dimension,
            "index_type": "FlatL2",
            "tile_size": PIXEL_TILE_SIZE,
            "built_at": time.time()
        }
        with open(PIXEL_INDEX_DIR / "metadata.json", "w", encoding="utf-8") as f:
            json.dump(metadata, f, ensure_ascii=False, indent=2)
        
        print(f"✅ Pixel FAISS index saved")
    else:
        print("⚠️  No valid pixel chunks to index")
else:
    print("ℹ️  No pixel chunks to index")

## 📦 Cell 8: Package & Download Results

In [ ]:
# ============================================================================
# Package indices into ZIP file
# ============================================================================

import shutil
import zipfile
from pathlib import Path
from datetime import datetime

INDICES_DIR = Path("/content/indices")
OUTPUT_DIR = Path("/content/output")
OUTPUT_DIR.mkdir(exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_zip_path = OUTPUT_DIR / f"rag_indices_{timestamp}.zip"

print(f"📦 Packaging indices...\n")
print(f"Source directory: {INDICES_DIR}")
print(f"Output file: {output_zip_path}")

# Create ZIP
with zipfile.ZipFile(output_zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(INDICES_DIR):
        for file in files:
            file_path = Path(root) / file
            arcname = file_path.relative_to(INDICES_DIR.parent)
            zipf.write(file_path, arcname)
            print(f"  ✓ {arcname}")

zip_size_mb = output_zip_path.stat().st_size / (1024 * 1024)
print(f"\n✅ ZIP file created: {zip_size_mb:.1f} MB")

In [ ]:
# ============================================================================
# Save to Google Drive
# ============================================================================

from pathlib import Path
import shutil

if SAVE_TO_DRIVE:
    drive_output_dir = Path("/content/drive/MyDrive/rag_indices_output")
    drive_output_dir.mkdir(parents=True, exist_ok=True)
    
    print("📤 Saving to Google Drive...")
    drive_output_zip = drive_output_dir / output_zip_path.name
    shutil.copy(output_zip_path, drive_output_zip)
    print(f"✅ Saved to Google Drive: {drive_output_zip}")
else:
    print("ℹ️  Skipping Google Drive save (SAVE_TO_DRIVE=False)")

In [ ]:
# ============================================================================
# Prepare for download
# ============================================================================

from google.colab import files
from pathlib import Path

print("\n📥 Preparing download...")
print(f"File: {output_zip_path.name}")
print(f"Size: {zip_size_mb:.1f} MB")
print(f"\n⏬ Starting download in 3 seconds...")
import time
time.sleep(1)

files.download(str(output_zip_path))
print("\n✅ Download complete!")

## 📊 Summary & Next Steps

In [ ]:
# ============================================================================
# Summary Report
# ============================================================================

print("""\n
╔════════════════════════════════════════════════════════════════╗
║            ✅ RAG INDEX GENERATION COMPLETE                    ║
╚════════════════════════════════════════════════════════════════╝

📊 STATISTICS:
""")

if text_chunks:
    print(f"  📚 Text RAG Index:")
    print(f"     • Chunks: {len(text_chunks)}")
    print(f"     • Embedding dimension: {text_embedder.get_sentence_embedding_dimension()}")
    print(f"     • Model: {TEXT_EMBEDDING_MODEL}")

if pixel_chunks:
    print(f"\n  🎨 Pixel RAG Index:")
    print(f"     • Tiles: {len(pixel_chunks)}")
    print(f"     • Embedding dimension: 384")
    print(f"     • Model: {PIXEL_EMBEDDING_MODEL}")

print(f"\n📦 OUTPUT:")
print(f"   • ZIP file: {output_zip_path.name}")
print(f"   • Size: {zip_size_mb:.1f} MB")
if SAVE_TO_DRIVE:
    print(f"   • Saved to: /rag_indices_output/")

print("\n📝 NEXT STEPS:")
print("""   1. Download the ZIP file from Colab
   2. Extract it in your AI-BreadBoard project:
      unzip rag_indices_*.zip -d data/rag_index/
   3. Update your RAG configuration to point to these indices
   4. Run your application and test the indices

⚠️  NOTES:
   • Text FAISS index uses L2 distance metric
   • Pixel embeddings are image feature vectors
   • All metadata saved in JSON format for easy inspection
   • Tiles stored with relative paths for portability
""")

print("\n✨ Happy searching!")